In [1]:
!python --version

Python 3.13.9


In [2]:
# Import necessary libraries
import os
from dotenv import load_dotenv
import google.generativeai as genai

# Load environment variables from .env file
load_dotenv()

# Configure the API key
api_key = os.getenv('GOOGLE_API_KEY')

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found. Please check your .env file."
    )

genai.configure(api_key=api_key)

print("✓ Google Gemini API configured successfully!")

✓ Google Gemini API configured successfully!


/Users/markgewhite/Documents/MyFiles/Projects/training/ztm/llm_web_apps/meal_plan/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def create_meals(ingredients, 
                 kcal=2000, 
                 exact_ingredients=False,
                 output_format='text', 
                 model='gemini-2.5-flash',
                 system_role='You are a skilled chef with expertise in cooking healthy meals.',
                 temperature=1,
                 extra=None):
    """
    Generate a daily meal plan using Google's Gemini AI.
    
    Parameters:
    - ingredients: List or string of available ingredients
    - kcal: Daily calorie limit (default 2000)
    - exact_ingredients: If True, use only provided ingredients
    - output_format: Format for output (text, markdown, etc.)
    - model: Gemini model to use (default: gemini-2.5-flash)
    - system_role: System instruction for the AI
    - temperature: Creativity level (0-2, default 1)
    - extra: Additional meal requirements
    
    Returns:
    - Text response with meal plan
    """
    
    # Construct the prompt
    prompt = f'''
    Create a healthy daily meal plan for breakfast lunch and dinner based on the following ingredients:
    ```{ingredients}```
    Your output should be in the {output_format} format.
    Follow the instructions below carefully.
    ### Instructions:
    1. {'Use ONLY the provided ingredients with salt, pepper and spices.' if exact_ingredients 
            else 'Feel free to adjust the provided ingredients as required for the meal.'}
    2. Specify the exact amount of each ingredient.
    3. Ensure the total daily calorie intake is below {kcal}.
    4. For each meal, explain each recipe step-by-step in clear and simple sentences. Use bullet points or numbers as appropriate.
    5. For each meal, specify the total number of calories and the number of servings.
    6. For each meal, provide a concise and descriptive title that summarises the main ingredients and flavours. The title should be sufficiently detailed for image generation to create an attractive image of the meal.
    7. For each recipe, indicate the preparation cooking and total time.
    {'8. If possible, the meals should be: ' + extra if extra else ''}
    
    Before answering, make sure that you have followed the instructions listed above.
    The last line of your answer should be a string that contains ONLY the titles of the recipes and nothing more with a comma between the main ingredients.
    '''
    
    # Initialize the Gemini model with system instruction
    gemini_model = genai.GenerativeModel(
        model_name=model,
        system_instruction=system_role
    )
    
    # Generate content with specified temperature
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(
            temperature=temperature,
        )
    )
    
    # Return the text response
    return response.text

In [4]:
# List available models
print("Available Gemini models:\n")
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(f"• {model.name}")
        print(f"  Display name: {model.display_name}")
        print(f"  Description: {model.description}")
        print()

Available Gemini models:

• models/gemini-2.5-pro-vtea-da-csi
  Display name: Gemini 2.5 Pro Preview with VTEA and DA CSI
  Description: Preview release (Nov 25th, 2025) of Gemini 2.5 Pro with VTEA and DA CSI

• models/gemini-2.5-pro-preview-03-25
  Display name: Gemini 2.5 Pro Preview 03-25
  Description: Gemini 2.5 Pro Preview 03-25

• models/gemini-2.5-flash
  Display name: Gemini 2.5 Flash
  Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.

• models/gemini-2.5-pro-preview-05-06
  Display name: Gemini 2.5 Pro Preview 05-06
  Description: Preview release (May 6th, 2025) of Gemini 2.5 Pro

• models/gemini-2.5-pro-preview-06-05
  Display name: Gemini 2.5 Pro Preview
  Description: Preview release (June 5th, 2025) of Gemini 2.5 Pro

• models/gemini-2.5-pro
  Display name: Gemini 2.5 Pro
  Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro

• models/gemini-2.0-flash-exp
  Display na

In [5]:
# Test the create_meals function
ingredients = """
- Chicken breast
- Broccoli
- Rice
- Eggs
- Tomatoes
- Onions
- Garlic
- Olive oil
"""

# Generate meal plan
meal_plan = create_meals(
    ingredients=ingredients,
    kcal=2000,
    exact_ingredients=False,
    output_format='markdown',
    temperature=0.8,
    extra='high in protein'
)

print(meal_plan)

Here is a healthy daily meal plan based on your provided ingredients, designed to be high in protein and under 2000 calories.

---

### Breakfast: Fluffy Scrambled Eggs with Diced Tomatoes and Sautéed Onions

A vibrant and protein-rich start to your day, featuring light and fluffy scrambled eggs mingled with sweet diced tomatoes and gently sautéed onions, offering a colorful and fresh appeal.

*   **Ingredients:**
    *   2 Large Eggs (approx. 110g)
    *   1/2 Medium Tomato (approx. 60g), diced
    *   1/4 Medium Onion (approx. 25g), finely diced
    *   1 teaspoon Olive Oil (5ml)
    *   Pinch of Salt and Black Pepper to taste
*   **Calories:** Approximately 240 kcal
*   **Servings:** 1
*   **Preparation Time:** 5 minutes
*   **Cooking Time:** 5 minutes
*   **Total Time:** 10 minutes

*   **Instructions:**
    1.  Heat 1 teaspoon of olive oil in a non-stick pan over medium heat.
    2.  Add the finely diced onion to the pan and sauté for 2-3 minutes until softened and translucent.
  